Code generated by Gemini.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install scikit-surprise
!pip uninstall numpy -y
!pip install numpy==1.26.4

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554969 sha256=a5d410c23ed9c961cf45fec5552f5ffb08e891607c0dc321261573de4baf5c94
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conf

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Load datasets
df_train = pd.read_json('/content/drive/MyDrive/SP26/CMPE 256/Group Project/beeradvocate_train.json', lines=True)

# Build an item profile based on aggregated review text and beer style
df_items = df_train[['beer/beerId', 'beer/name', 'beer/style', 'review/text']].dropna()

# Aggregate text per beer to build item profiles
item_profiles = df_items.groupby(['beer/beerId', 'beer/name', 'beer/style'])['review/text'].apply(lambda x: ' '.join(x)).reset_index()

# Combine style and review text for rich content
item_profiles['content'] = item_profiles['beer/style'] + " " + item_profiles['review/text']

print(f"Created profiles for {len(item_profiles)} beers.")


Created profiles for 45265 beers.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
from tqdm import tqdm

# 1. Compute TF-IDF matrix for items
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(item_profiles['content'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

# 2. Create helper dictionaries for fast lookups
beer_id_to_idx = {beer_id: idx for idx, beer_id in enumerate(item_profiles['beer/beerId'])}
global_mean = df_train['review/overall'].mean()

# Precompute user histories (which beers they rated and what rating they gave)
print("Building user history mappings...")
user_history = df_train.dropna(subset=['review/profileName', 'beer/beerId', 'review/overall'])\
    .groupby('review/profileName')\
    .apply(lambda x: dict(zip(x['beer/beerId'], x['review/overall'])), include_groups=False).to_dict()

user_means = df_train.groupby('review/profileName')['review/overall'].mean().to_dict()

# 3. Rating prediction function
def predict_rating(user, beer_id):
    # If we don't know the beer, return user's average or global average
    if beer_id not in beer_id_to_idx:
        return user_means.get(user, global_mean)

    # If we don't know the user, return global average
    if user not in user_history or not user_history[user]:
        return global_mean

    target_idx = beer_id_to_idx[beer_id]
    target_vector = tfidf_matrix[target_idx]

    user_ratings = user_history[user]
    valid_rated_beers = [b for b in user_ratings.keys() if b in beer_id_to_idx]

    if not valid_rated_beers:
        return user_means.get(user, global_mean)

    rated_indices = [beer_id_to_idx[b] for b in valid_rated_beers]
    rated_vectors = tfidf_matrix[rated_indices]

    # Compute cosine similarities between target beer and user's rated beers
    sims = linear_kernel(target_vector, rated_vectors).flatten()
    ratings_array = np.array([user_ratings[b] for b in valid_rated_beers])

    # Only consider positive similarities
    mask = sims > 0
    if not np.any(mask):
        return user_means.get(user, global_mean)

    weighted_sum = np.sum(sims[mask] * ratings_array[mask])
    sum_sims = np.sum(sims[mask])

    if sum_sims == 0:
        return user_means.get(user, global_mean)

    prediction = weighted_sum / sum_sims
    return max(1.0, min(5.0, prediction)) # Clip between 1 and 5

print("Prediction function ready.")

TF-IDF Matrix shape: (45265, 5000)
Building user history mappings...
Prediction function ready.


In [ ]:
# 4. Evaluate using RMSE and MAE
def evaluate_predictions(df_eval, sample_size=2000):
    df_eval_clean = df_eval.dropna(subset=['review/profileName', 'beer/beerId', 'review/overall'])

    # Use a sample for faster evaluation demonstration, or None for full dataset
    if sample_size and len(df_eval_clean) > sample_size:
        df_eval_sample = df_eval_clean.sample(sample_size, random_state=42)
    else:
        df_eval_sample = df_eval_clean

    y_true = []
    y_pred = []

    for _, row in tqdm(df_eval_sample.iterrows(), total=len(df_eval_sample)):
        y_true.append(row['review/overall'])
        y_pred.append(predict_rating(row['review/profileName'], row['beer/beerId']))

    rmse = sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

# Load val and test sets
df_val = pd.read_json('/content/drive/MyDrive/SP26/CMPE 256/Group Project/beeradvocate_val.json', lines=True)
df_test = pd.read_json('/content/drive/MyDrive/SP26/CMPE 256/Group Project/beeradvocate_test.json', lines=True)

print("Evaluating on Validation Set...")
val_rmse, val_mae = evaluate_predictions(df_val, sample_size=4000)
print(f"Validation RMSE: {val_rmse:.4f}, MAE: {val_mae:.4f}")

print("\nEvaluating on Test Set...")
test_rmse, test_mae = evaluate_predictions(df_test, sample_size=4000)
print(f"Test RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}")

Evaluating on Validation Set...


100%|██████████| 4000/4000 [01:26<00:00, 46.17it/s]


Validation RMSE: 0.6773, MAE: 0.4912

Evaluating on Test Set...


100%|██████████| 4000/4000 [01:15<00:00, 53.12it/s]

Test RMSE: 0.7064, MAE: 0.5172
